# Embedding baseline with human-labeled examples

Each tagged transcript is encoded as a single vector with `sentence-transformers/bert-base-nli-mean-tokens`. Each level is represented by the mean embedding of its labeled examples, and every transcript gets the level of the nearest mean.

**Data**: a csv with one row per transcript, its tagged text (`Transcript`) and the level given by humans (`ability_label_name`: Low / Medium / High).

**Protocol** (identical in `human_fgw.ipynb` and `human_embeddings.ipynb`):
* stratified $k$-fold cross-validation on the human labels;
* for each fold, $m$ labeled transcripts per level are drawn from the other folds (all of them with `M = None`), and the transcripts of the fold are classified with them;
* the folds and examples are saved in `splits/` and reloaded, so both methods use exactly the same ones;
* scores against the human level: accuracy, macro-F1, quadratic-weighted Cohen's $\kappa$ with a 95% bootstrap CI over transcripts;
* robustness: the whole cross-validation is repeated with `N_REPEATS` different random splits (mean ± SD).

In [ ]:
import os
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

sys.path.append(os.path.realpath("../lib"))
from evaluation import (LEVELS, evaluate, load_labeled_transcripts, load_or_create_split, plot_confusion,
                        scores, summarize)

## Parameters

In [ ]:
LABELS_FILE = "human_labels.csv"
TEXT_COLUMN = "Transcript"
LABEL_COLUMN = "ability_label_name"
DIMENSION = "ability"           # names the splits and outputs

SPLIT_DIR = "splits"            # shared with the other notebook
OUTPUT_DIR = "results"

M = 10                          # labeled examples per level (None: all the transcripts of the other folds)
N_FOLDS = 5
SEED = 0                        # split of the main results
N_REPEATS = 5                   # random splits for the robustness check (seeds SEED, SEED + 1, ...)

EMBEDDING_MODEL = "sentence-transformers/bert-base-nli-mean-tokens"
METRIC = "cosine"               # "cosine" or "euclidean", as in embedding_baseline.ipynb

os.makedirs(OUTPUT_DIR, exist_ok=True)

## Human-labeled transcripts

In [ ]:
df = load_labeled_transcripts(LABELS_FILE, TEXT_COLUMN, LABEL_COLUMN)
counts = df["level"].value_counts().reindex(LEVELS, fill_value=0)
print(f"{len(df)} transcripts")
if M is not None:
    needed = int(np.ceil(M * N_FOLDS / (N_FOLDS - 1)))
    print(f"m = {M} examples per level from {N_FOLDS - 1} folds out of {N_FOLDS}: "
          f"about {needed} transcripts per level are needed")
counts.to_frame("transcripts")

## Embeddings

In [ ]:
model = SentenceTransformer(EMBEDDING_MODEL)
X = model.encode(df["text"].tolist(), batch_size=64, show_progress_bar=True)
if METRIC == "cosine":  # cosine distance = Euclidean distance between normalized vectors
    X = X / np.linalg.norm(X, axis=1, keepdims=True)
vectors = dict(zip(df["id"], X))

## Nearest mean embedding

In [ ]:
def classify(folds, examples):
    # Predicted level of every transcript, from the examples of its own fold.
    predictions = []
    for fold in sorted(folds["fold"].unique()):
        test = folds[folds["fold"] == fold]
        support = examples[examples["fold"] == fold]
        centroids = np.array([np.mean([vectors[i] for i in support.loc[support["level"] == level, "id"]], axis=0)
                              for level in LEVELS])
        X_test = np.array([vectors[i] for i in test["id"]])
        distances = np.linalg.norm(X_test[:, None, :] - centroids[None, :, :], axis=2)
        pred = pd.DataFrame({"id": test["id"], "fold": fold, "true": test["level"],
                             "pred": np.array(LEVELS)[distances.argmin(axis=1)]})
        for k, level in enumerate(LEVELS):
            pred[f"distance_{level}"] = distances[:, k]
        predictions.append(pred)
    return pd.concat(predictions, ignore_index=True)

## Classification (main split)

In [ ]:
SETTING = f"m = {M}" if M is not None else "all examples"
folds, examples = load_or_create_split(df, DIMENSION, f"human_seed{SEED}", SPLIT_DIR, M, N_FOLDS, SEED)
pred = classify(folds, examples)
pred.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_predictions_{DIMENSION}.csv"), index=False)

results = evaluate({(DIMENSION, SETTING): pred}, method="embeddings", reference="human")
results.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_results_{DIMENSION}.csv"), index=False)
summarize(results)

All folds pooled, with the 95% bootstrap CI of $\kappa$:

In [ ]:
pooled = results[results["fold"] == "all"].drop(columns="fold").copy()
pooled["kappa [95% CI]"] = [f"{k:.2f} [{lo:.2f}, {hi:.2f}]" for k, lo, hi in
                            zip(pooled["kappa"], pooled["kappa_ci_low"], pooled["kappa_ci_high"])]
pooled.round(3)

In [ ]:
plot_confusion({(DIMENSION, SETTING): pred}, "human", os.path.join(OUTPUT_DIR, f"human_embedding_confusion_{DIMENSION}.png"))
plt.show()

## Robustness over random splits

The same cross-validation with `N_REPEATS` different splits (the folds and examples of each are saved too).

In [ ]:
repeats = []
for seed in range(SEED, SEED + N_REPEATS):
    folds_r, examples_r = load_or_create_split(df, DIMENSION, f"human_seed{seed}", SPLIT_DIR, M, N_FOLDS, seed)
    pred_r = pred if seed == SEED else classify(folds_r, examples_r)
    repeats.append({"seed": seed, **scores(pred_r)})
repeats = pd.DataFrame(repeats)
repeats.to_csv(os.path.join(OUTPUT_DIR, f"human_embedding_repeats_{DIMENSION}.csv"), index=False)
display(repeats.round(3))
repeats[["accuracy", "macro_f1", "kappa"]].agg(["mean", "std"]).round(3)